### Reddit Subreddit Rules Evolution for Health Crises

#### Overview
This module collects historical moderation rules from health-related subreddits to study how rules evolved over time during public health crises. The focus is on identifying when rules were introduced, modified, or removed, and how these changes aligned with major events such as the COVID-19 pandemic (2020–2023) and the mpox outbreak (2022).

#### Data Collected
The data is limited to:
- Subreddit rules and guidelines  
- Pinned moderator announcements  
- Policy clarifications issued by moderators  
- Timestamps of rule updates  

#### Objective
The goal is to construct a temporal map of rule evolution to analyze how subreddit governance adapted in response to health misinformation and shifting public health contexts.


In [15]:
import re
from pathlib import Path


In [16]:
def extract_date(filename):
    """
    Extracts YYYY-MM-DD from filenames like rules2020-03-31.json
    """
    m = re.search(r"rules(\d{4}-\d{2}-\d{2})", filename)
    return m.group(1) if m else None


In [17]:
import json
import pandas as pd


In [22]:
import json
import pandas as pd
import re
from pathlib import Path
# --- Loader ---
def load_rules_file(path):
    with open(path, "r", encoding="utf-8") as f:
        try:
            data = json.load(f)
        except json.JSONDecodeError:
            print(f"❌ Invalid JSON: {path}")
            return pd.DataFrame()

    if not data or "title" not in data or "description" not in data:
        print(f"⚠️ Empty or missing keys: {path}")
        return pd.DataFrame()

    rows = []
    for idx, title in data["title"].items():
        desc = data["description"].get(idx, "")
        m = re.search(r"rule\s*(\d+)", title.lower())
        rule_id = f"rule_{m.group(1)}" if m else f"rule_unknown_{idx}"

        rows.append({
            "rule_id": rule_id,
            "title": title.strip(),
            "description": desc.strip()
        })

    return pd.DataFrame(rows)


In [19]:
import glob


In [26]:
import re

def tokenize(text):
    """Simple tokenizer: lowercase, split by non-word chars"""
    if pd.isna(text) or not text:
        return set()
    return set(re.findall(r'\w+', text.lower()))

def compare_descriptions(prev_desc, curr_desc):
    prev_words = tokenize(prev_desc)
    curr_words = tokenize(curr_desc)

    added = curr_words - prev_words
    deleted = prev_words - curr_words
    changed = bool(added or deleted)

    return changed, added, deleted


In [27]:
# --- Label type of change ---
def change_type(row):
    if pd.isna(row["prev_title"]):
        return "new_rule"
    if row["title_changed"] and row["description_changed"]:
        return "title+description"
    if row["title_changed"]:
        return "title_changed"
    if row["description_changed"]:
        return "description_changed"
    return "none"


In [29]:
import glob


# --- Load all snapshots ---
all_snapshots = []

for file in sorted(glob.glob("./data/rules/rules2020-*.json")):  # adjust pattern to match your files
    date_match = re.search(r"rules(\d{4}-\d{2}-\d{2})", file)
    if not date_match:
        continue
    date = date_match.group(1)

    df = load_rules_file(file)
    if df.empty:
        continue

    df["date"] = date
    all_snapshots.append(df)




⚠️ Empty or missing keys: ./data/rules\rules2020-10-14.json


In [30]:
# --- Combine snapshots ---
rules_df = pd.concat(all_snapshots, ignore_index=True)
rules_df = rules_df.sort_values(["rule_id", "date"])

# --- Track changes ---
rules_df["prev_title"] = rules_df.groupby("rule_id")["title"].shift(1)
rules_df["prev_description"] = rules_df.groupby("rule_id")["description"].shift(1)

# Title change
rules_df["title_changed"] = rules_df["title"] != rules_df["prev_title"]

# Initialize columns for sub-rule tracking
rules_df["description_changed"] = False
rules_df["description_added"] = ""
rules_df["description_deleted"] = ""
rules_df["description_change_type"] = ""

# Helper function to compare descriptions
import re
def tokenize(text):
    if pd.isna(text) or not text:
        return set()
    return set(re.findall(r"\w+", text.lower()))

def compare_descriptions(prev_desc, curr_desc):
    prev_words = tokenize(prev_desc)
    curr_words = tokenize(curr_desc)
    added = curr_words - prev_words
    deleted = prev_words - curr_words
    changed = bool(added or deleted)
    return changed, added, deleted

# Apply description comparison row by row
for idx, row in rules_df.iterrows():
    if pd.isna(row["prev_title"]):
        # New rule
        rules_df.at[idx, "description_change_type"] = "new_rule"
    elif row["title_changed"]:
        # Title changed → consider full rewrite
        rules_df.at[idx, "description_change_type"] = "title_changed"
    else:
        # Same title → compare descriptions
        changed, added, deleted = compare_descriptions(row["prev_description"], row["description"])
        rules_df.at[idx, "description_changed"] = changed
        rules_df.at[idx, "description_added"] = ", ".join(added) if added else ""
        rules_df.at[idx, "description_deleted"] = ", ".join(deleted) if deleted else ""

        if changed:
            if added and deleted:
                rules_df.at[idx, "description_change_type"] = "updated"
            elif added:
                rules_df.at[idx, "description_change_type"] = "added"
            elif deleted:
                rules_df.at[idx, "description_change_type"] = "deleted"
        else:
            rules_df.at[idx, "description_change_type"] = "none"

# Overall change vs previous
rules_df["changed_vs_prev"] = rules_df["title_changed"] | rules_df["description_changed"]

# Optional: overall change type (title or description)
def change_type(row):
    if pd.isna(row["prev_title"]):
        return "new_rule"
    if row["title_changed"]:
        return "title_changed"
    return row["description_change_type"]

rules_df["change_type"] = rules_df.apply(change_type, axis=1)

# --- Save final dataset ---
rules_df[[
    "date", "rule_id", "title", "description",
    "description_change_type", "description_added", "description_deleted",
    "changed_vs_prev", "change_type"
]].to_csv("rules_change_tracking.csv", index=False)

print("✅ Done! Dataset ready: rules_change_tracking.csv")


✅ Done! Dataset ready: rules_change_tracking.csv
